## ISOM4030 Lab 7.2 - Model Tuning

### Overview

The process of selecting the right set of hyperparameters for your machine learning (ML) application is called *hyperparameter tuning* or *hypertuning*.

Hyperparameters are the variables that govern the training process and the topology of an ML model. These variables remain constant over the training process and directly impact the performance of your ML program.

In this lab, we will introduce various model tuning techniques, including batch normalization, dropout layers, optimizers with momentum, and learning rate decay schedules. These techniques are fundamental for optimizing and improving the performance of machine learning models.

Additionally, we will explore the Keras Tuner, a powerful tool that assists in selecting the optimal set of hyperparameters for your TensorFlow program. Through this lab, you will gain practical insights into these tuning techniques and learn how to leverage the Keras Tuner effectively.



#### Setup

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, losses, optimizers, callbacks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization

Install and import the Keras Tuner.

In [ ]:
!pip install -q -U keras-tuner # -q is short for quiet; -U is short for upgrade

ERROR: Invalid requirement: '#'


In [ ]:
import keras_tuner as kt

Using TensorFlow backend


#### Download and prepare the dataset

In this tutorial, you will use the Keras Tuner to find the best hyperparameters for a machine learning model that classifies images of clothing from the [Fashion MNIST dataset](https://github.com/zalandoresearch/fashion-mnist).

In [ ]:
(img_train, label_train), (img_test, label_test) = keras.datasets.fashion_mnist.load_data()

In [ ]:
# normalize pixel values between 0 and 1
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0

#### Warm up: Build the model with Sequential API.

In [ ]:
# Model built with Sequential API

model_init = Sequential([




                ])
# Print a summary of the network to examine model architecture
print(model_init.summary())


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense (Dense)               (None, 128)               100480    
                                                                 
 dense_1 (Dense)             (None, 10)                1290      
                                                                 
Total params: 101,770
Trainable params: 101,770
Non-trainable params: 0
_________________________________________________________________
None


### BatchNormalization Layer
A Batch Normalization (BN) layer is implemented as [`tf.keras.layers.BatchNormalization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization)

In a BN layer, there are 4 parameters for each node in the previous dense layer: `gamma`, `beta`, `moving_mean`, and `moving_variance`.

`Gamma (scale)` and `beta (offset)` are trainable parameters. `moving_mean` and `moving_variance` are non-trainable parameters (used during prediction).

With a dense layer of 128 nodes:

- Trainable parameters: 128 (gamma) + 128 (beta) = 256
- Non-trainable parameters: 128 (moving_mean) + 128 (moving_variance) = 256

Total parameters in BN layer: 256 (trainable) + 256 (non-trainable) = 512


Benefits of BN layer:
- Improve training stability, allow higher learning rate
- Accelerate training convergence, enhance gradient flow
- Regularization Effect

<font color = 'red'><b>Note</b></font>: There are two opinions for where the BN layer should be placed in the model architecture — before and after activation.

In [ ]:
# Add BN layer after activation function
model_init = Sequential([
            layers.Flatten(input_shape=(28, 28)),
            layers.Dense(128, activation='relu'),
            layers.BatchNormalization(),    # By default: trainable = True
            layers.Dense(10)
])

model_init.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_1 (Flatten)         (None, 784)               0         
                                                                 
 dense_2 (Dense)             (None, 128)               100480    
                                                                 
 batch_normalization (BatchN  (None, 128)              512       
 ormalization)                                                   
                                                                 
 dense_3 (Dense)             (None, 10)                1290      
                                                                 
Total params: 102,282
Trainable params: 102,026
Non-trainable params: 256
_________________________________________________________________


In [ ]:
# Add BN layer before activation function
model_init = Sequential([
            layers.Flatten(input_shape=(28, 28)),
            layers.Dense(128),
            layers.BatchNormalization(),    # By default: trainable = True
            layers.ReLU(),
            layers.Dense(10)
])
model_init.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_2 (Flatten)         (None, 784)               0         
                                                                 
 dense_4 (Dense)             (None, 128)               100480    
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                                                 
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dense_5 (Dense)             (None, 10)                1290      
                                                                 
Total params: 102,282
Trainable params: 102,026
Non-trainable params: 256
______________________________________________

### Dropout Layer
Last time, we mentioned the dropout layer - [`tf.keras.layers.Dropout()`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout). The Dropout layer randomly sets input units to 0 with a frequency of rate at each step during training time, which helps prevent overfitting. Inputs not set to 0 are scaled up by 1/(1 - rate) such that the sum over all inputs is unchanged.


<font color = 'red'><b>Note</b></font>: Do not add BN layer after dropout.


In [ ]:
model_init = Sequential([
            layers.Flatten(input_shape=(28, 28)),
            layers.Dense(128),
            layers.BatchNormalization(),    # By default: trainable = True
            layers.ReLU(),
            layers.Dropout(0.2),
            layers.Dense(10)
])
model_init.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_3 (Flatten)         (None, 784)               0         
                                                                 
 dense_6 (Dense)             (None, 128)               100480    
                                                                 
 batch_normalization_2 (Batc  (None, 128)              512       
 hNormalization)                                                 
                                                                 
 re_lu_1 (ReLU)              (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_7 (Dense)             (None, 10)                1290      
                                                      

### Optimizer selection
SGD is a simple optimization algorithm, Adam offer adaptive learning rates and additional techniques to accelerate convergence. Adam, in particular, combines the benefits of both SGD and RMSprop, making it a popular choice in many deep learning applications.



#### Default Values - For Your Reference Only

<font color = 'red'><b>Default values in key functions</b> </font>

**SGD** vs **Adam**:

- [`SGD()` constructor](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/SGD)

```python
tf.keras.optimizers.experimental.SGD(
    learning_rate=0.01, ...
)
```


- [`Adam()` constructor](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam)

```python
tf.keras.optimizers.Adam(
    learning_rate=0.001, ...
)

```


### EarlyStopping

Create a callback to stop training early after reaching a certain value for the validation loss. Specify the `callbacks` by the defined earlystopping criteria while the model is trained with `.fit()`.

*patience=2*: This means the training will stop after 2 consecutive epochs where the validation loss does not improve.

In [ ]:
stop_early = callbacks.EarlyStopping(monitor='val_loss', patience=2)

#### <font color='steelblue' > Exercise </font>
Build a neural network model in TensorFlow with the given specifications:

- Flatten layer with input shape: (28, 28)
- Hidden layer: One hidden layer with 128 nodes, activated by ReLU
- Batch Normalization (BN) layer: Implemented for improved training stability
- Dropout layer: 20%
- Optimizer: Adam optimizer with default learning rate
- Batch size: 32
- No. of epochs: 10
- EarlyStopping callback: Included to monitor validation loss
- 20% data split on training set for evaluation

In [ ]:
# Step 1: build the model
model_ex1 = Sequential([



                ])

# Step 2: Compile the model (Specify the loss function and optimizer to use)
model_ex1.compile(optimizer=______________,
              loss=tf.keras.losses._________________________________(_________________),
              metrics=['accuracy'])

# Step 3: Train the model (Specify batch size and number of epochs)
model_ex_output = model_ex1.______(__________, __________, __________, __________, validation_split=_____,  callbacks=_________)

Epoch 1/10
1500/1500 [==============================] - 3s 2ms/step - loss: 0.5212 - accuracy: 0.8166 - val_loss: 0.4347 - val_accuracy: 0.8354
Epoch 2/10
1500/1500 [==============================] - 2s 2ms/step - loss: 0.4096 - accuracy: 0.8517 - val_loss: 0.3551 - val_accuracy: 0.8727
Epoch 3/10
1500/1500 [==============================] - 2s 1ms/step - loss: 0.3752 - accuracy: 0.8628 - val_loss: 0.3566 - val_accuracy: 0.8717
Epoch 4/10
1500/1500 [==============================] - 2s 1ms/step - loss: 0.3529 - accuracy: 0.8709 - val_loss: 0.3378 - val_accuracy: 0.8773
Epoch 5/10
1500/1500 [==============================] - 2s 2ms/step - loss: 0.3343 - accuracy: 0.8792 - val_loss: 0.3318 - val_accuracy: 0.8811
Epoch 6/10
1500/1500 [==============================] - 2s 2ms/step - loss: 0.3176 - accuracy: 0.8839 - val_loss: 0.3370 - val_accuracy: 0.8773
Epoch 7/10
1500/1500 [==============================] - 2s 2ms/step - loss: 0.3108 - accuracy: 0.8861 - val_loss: 0.3417 - val_accuracy:

### Keras Tuner the model tuning

When you build a model for hypertuning, you also define the hyperparameter search space in addition to the model architecture. The model you set up for hypertuning can be perceived as a *hypermodel*.

You can define a hypermodel by using a model builder function, which defines inline the hyperparameters you want to tune and returns a compiled model.

In [ ]:
def model_builder(hp):
    # Tune the number of units in the 1st Dense layer
    # Choose an optimal value between 32-512
    hp_units_1 = hp.Int('units_1', min_value=32, max_value=512, step=32)
    hp_units_2 = hp.Int('units_2', min_value=32, max_value=512, step=32)

    model = keras.Sequential()
    model.add(Flatten(input_shape=(28, 28)))
    model.add(Dense(units=hp_units_1, activation='relu'))

    if hp.Boolean("bn_1"):
        model.add(BatchNormalization())

    model.add(Dense(units=hp_units_2, activation='relu'))
    if hp.Boolean("bn_2"):
        model.add(BatchNormalization())
    model.add(Dense(10))

    # Tune the learning rate for the optimizer
    # Choose an optimal value from 0.01, 0.001, or 0.0001
    hp_learning_rate = hp.Float("learning_rate", min_value=0.001, max_value=10,
                              step=3, sampling="log")   # log - nonlinear: [0.001, 0.01, 0.1, 1, 10]

    hp_optimizer = hp.Choice('optimizer', values = ['adam', 'sgd'], ordered = False)

    if hp_optimizer == 'adam':
        opt_algo = optimizers.Adam(learning_rate=hp_learning_rate)
    else:
        opt_algo = optimizers.SGD(learning_rate=hp_learning_rate)

    model.compile(optimizer=opt_algo,
                    loss=losses.SparseCategoricalCrossentropy(from_logits=True),
                    metrics=['accuracy'])

    return model

you can find the way to tune the number of layers here:
https://keras.io/guides/keras_tuner/getting_started/

## Instantiate the tuner and perform hypertuning

Instantiate the tuner to perform the hypertuning. The Keras Tuner has 5 tuners available - `RandomSearch`, `Hyperband`, `BayesianOptimization`, and `Sklearn`.

In this tutorial, you use the [`RandomSearch` tuner](https://keras.io/api/keras_tuner/tuners/random/#randomsearch-class).

To instantiate the Hyperband tuner, you must specify the `hypermodel`, the `objective` to optimize, and the total number of experiments (model configurations) to test at most (`max_trials`).

In [ ]:
#! rm -r my_dir

In [ ]:
tuner = kt.RandomSearch(model_builder,
                        objective='val_accuracy',
                        max_trials=10,
                        seed=1,                # to make sure the result is reproducible
                        #directory='my_dir',   # create a folder to store the model
                        project_name='intro_to_kt')

Reloading Tuner from .\intro_to_kt\tuner0.json


You can print a summary of the search space:

In [ ]:
tuner.search_space_summary()

Search space summary
Default search space size: 8
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
units_2 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
weight_init (Choice)
{'default': 'he_uniform', 'conditions': [], 'values': ['he_uniform', 'he_normal'], 'ordered': False}
bn_1 (Boolean)
{'default': False, 'conditions': []}
bn_2 (Boolean)
{'default': False, 'conditions': []}
learning_rate (Float)
{'default': 0.001, 'conditions': [], 'min_value': 0.001, 'max_value': 10.0, 'step': 3, 'sampling': 'log'}
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'sgd'], 'ordered': False}
momentum (Float)
{'default': 0.6, 'conditions': [], 'min_value': 0.6, 'max_value': 0.9, 'step': None, 'sampling': 'linear'}


Run the hyperparameter search. The arguments for the search method are the same as those used for `tf.keras.model.fit` in addition to the callback above.

In [ ]:
tuner.search(img_train, label_train, batch_size=256, epochs=5, validation_split=0.2, callbacks=[stop_early])

To check the summary for the hypertuning job, we simply use `.results_summary()`:

In [ ]:
tuner.results_summary(5)

Results summary
Results in .\intro_to_kt
Showing 5 best trials
Objective(name="val_accuracy", direction="max")

Trial 04 summary
Hyperparameters:
units_1: 288
units_2: 288
weight_init: he_uniform
bn_1: False
bn_2: True
learning_rate: 0.081
optimizer: sgd
momentum: 0.6
Score: 0.8648333549499512

Trial 06 summary
Hyperparameters:
units_1: 224
units_2: 160
weight_init: he_normal
bn_1: False
bn_2: True
learning_rate: 0.001
optimizer: adam
momentum: 0.6730553288128011
Score: 0.8608333468437195

Trial 01 summary
Hyperparameters:
units_1: 128
units_2: 256
weight_init: he_normal
bn_1: False
bn_2: False
learning_rate: 0.009000000000000001
optimizer: adam
Score: 0.8577499985694885

Trial 05 summary
Hyperparameters:
units_1: 192
units_2: 352
weight_init: he_normal
bn_1: False
bn_2: False
learning_rate: 0.027
optimizer: sgd
momentum: 0.611565551801214
Score: 0.8550000190734863

Trial 09 summary
Hyperparameters:
units_1: 384
units_2: 352
weight_init: he_normal
bn_1: False
bn_2: False
learning_rate:

In [ ]:
# Get the optimal hyperparameters
# optimal_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

# Get the best k sets of hyperparameters
best_k_hps = tuner.get_best_hyperparameters(num_trials=5)  # e.g., k = 5
for best_hp in best_k_hps:
  print(f"units: {best_hp.get('units_1')}; learning_rate: {best_hp.get('learning_rate')}")

units: 288; learning_rate: 0.081
units: 224; learning_rate: 0.001
units: 128; learning_rate: 0.009000000000000001
units: 192; learning_rate: 0.027
units: 384; learning_rate: 0.009000000000000001


When search is over, you can retrieve the best model(s). The model is saved at its best performing epoch evaluated on the validation_data.

In [ ]:
# Get the top 2 models.
models = tuner.get_best_models(num_models=2)
best_model = models[0]
# Build the model
# only needed for `Sequential` without specified `input_shape`.
# best_model.build(input_shape=(None, 28, 28))
best_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense (Dense)               (None, 288)               226080    
                                                                 
 dense_1 (Dense)             (None, 288)               83232     
                                                                 
 batch_normalization (BatchN  (None, 288)              1152      
 ormalization)                                                   
                                                                 
 dense_2 (Dense)             (None, 10)                2890      
                                                                 
Total params: 313,354
Trainable params: 312,778
Non-trainable params: 576
________________________________________________

## Train the model



Find the optimal number of epochs to train the model with the hyperparameters obtained from the search.

In [ ]:
# Build the model with the optimal hyperparameters and train it on the data for 50 epochs
model = model_builder(best_k_hps[0])

model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_1 (Flatten)         (None, 784)               0         
                                                                 
 dense_3 (Dense)             (None, 288)               226080    
                                                                 
 dense_4 (Dense)             (None, 288)               83232     
                                                                 
 batch_normalization_1 (Batc  (None, 288)              1152      
 hNormalization)                                                 
                                                                 
 dense_5 (Dense)             (None, 10)                2890      
                                                                 
Total params: 313,354
Trainable params: 312,778
Non-trainable params: 576
______________________________________________

In [ ]:
history = model.fit(img_train, label_train, batch_size=256, epochs=50, validation_split=0.2)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % (best_epoch,))

Epoch 1/50
188/188 [==============================] - 1s 5ms/step - loss: 0.5078 - accuracy: 0.8222 - val_loss: 0.4770 - val_accuracy: 0.8442
Epoch 2/50
188/188 [==============================] - 1s 4ms/step - loss: 0.3826 - accuracy: 0.8625 - val_loss: 0.4772 - val_accuracy: 0.8348
Epoch 3/50
188/188 [==============================] - 1s 4ms/step - loss: 0.3470 - accuracy: 0.8749 - val_loss: 0.4478 - val_accuracy: 0.8363
Epoch 4/50
188/188 [==============================] - 1s 4ms/step - loss: 0.3210 - accuracy: 0.8831 - val_loss: 0.4399 - val_accuracy: 0.8433
Epoch 5/50
188/188 [==============================] - 1s 4ms/step - loss: 0.3035 - accuracy: 0.8868 - val_loss: 0.4348 - val_accuracy: 0.8519
Epoch 6/50
188/188 [==============================] - 1s 4ms/step - loss: 0.2855 - accuracy: 0.8953 - val_loss: 0.4389 - val_accuracy: 0.8417
Epoch 7/50
188/188 [==============================] - 1s 4ms/step - loss: 0.2759 - accuracy: 0.8982 - val_loss: 0.4108 - val_accuracy: 0.8605
Epoch 

To finish this tutorial, evaluate the hypermodel on the test data.

In [ ]:
eval_result = model.evaluate(img_test, label_test)
print("[test loss, test accuracy]:", eval_result)

313/313 [==============================] - 0s 1ms/step - loss: 0.5117 - accuracy: 0.8830
[test loss, test accuracy]: [0.5116829872131348, 0.8830000162124634]


The `my_dir/intro_to_kt` directory contains detailed logs and checkpoints for every trial (model configuration) run during the hyperparameter search. If you re-run the hyperparameter search, the Keras Tuner uses the existing state from these logs to resume the search. To disable this behavior, pass an additional `overwrite=True` argument while instantiating the tuner.

## Summary

In this tutorial, you learned how to use the Keras Tuner to tune hyperparameters for a model. To learn more about the Keras Tuner, check out these additional resources:

* [Keras Tuner - Hyperparameters](https://keras.io/api/keras_tuner/hyperparameters/)
* [Keras Tuner website](https://keras-team.github.io/keras-tuner/)

Also check out the [HParams Dashboard](https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams) in TensorBoard to interactively tune your model hyperparameters.